## 19. 系统提示定制：四条路线

> 来源：[Modifying system prompts](https://code.claude.com/docs/en/agent-sdk/modifying-system-prompts)

`system_prompt` 有四种写法，先记住一个反差：**SDK 默认的 system prompt 是"最小版"，只覆盖 tool calling，不是 CLI 的完整 Claude Code prompt**——而文件系统配置（§20「setting_sources」）默认却是全加载。从 `claude -p` 迁移想要一致行为，必须显式用 preset。

| 写法 | 得到什么 |
|---|---|
| 不设 | 最小默认：只有 tool-calling 支持，无编码规范/回复风格/安全指令 |
| `system_prompt={"type": "preset", "preset": "claude_code"}` | CLI 同款完整 prompt（tool 指令、代码风格、语气、安全、环境上下文） |
| preset + `"append": "..."` | 全保留再追加自己的规则——**风险最低的定制**，做编码类产品首选 |
| `system_prompt="自定义字符串"` | 只有自定义字符串本身的内容；tool 指导和安全指令都得自己补。适合非编码类 agent，或需要给 agent 换一套身份、口吻的场景（比如同一后端在不同产品入口里扮演不同角色） |

两个关键原理：

- **CLAUDE.md 不进 system prompt**——SDK 把它注入 conversation 作为项目上下文，与任何 system_prompt 配置兼容；它的加载由 `setting_sources` 控制，跟 preset 无关。眼见为实的验证方法：翻 session transcript（`~/.claude/projects/<encoded-cwd>/<session_id>.jsonl`，路径规则见 §13.2「存储位置」），CLAUDE.md 全文出现在**首条 user 消息**注入的上下文里，system prompt 条目里没有它。
- **preset 里嵌了 per-session 动态段**（工作目录、平台、shell、git 状态），不同目录的两个 session 因此无法共享 prompt cache。设 `"exclude_dynamic_sections": True`（SDK ≥ 0.1.58，仅 preset 形式有效）把动态段挪到首条 user message，让相同配置跨用户跨机器命中同一条 cache。

```python
options = ClaudeAgentOptions(
    system_prompt={
        "type": "preset",
        "preset": "claude_code",
        "append": "Always include detailed docstrings and type hints in Python code.",
        "exclude_dynamic_sections": True,  # 跨 session 共享 prompt cache
    },
    setting_sources=["project"],  # CLAUDE.md 由这里控制，不由 preset 控制
)
```

第四条路线是 **output styles**（`~/.claude/output-styles/*.md` 或项目 `.claude/output-styles/`）：文件化的 system prompt 修改。一个示例文件看清两个行为开关：

```markdown
---
name: Tutor
description: 教学导向的回复风格
keep-coding-instructions: true   # true=叠加在 preset 软件工程指令之上；省略则默认整段替换
---

每次修改代码后，用一句话解释为什么这样改，再给出可运行的验证命令。
```

Python SDK 没有编程选择 output style 的选项，只能靠 settings 文件激活——纯代码部署直接用 preset + append 替代。

四种方式的定位：CLAUDE.md 管**项目级持久约定**；output style 管**跨项目的回复风格改写**；preset + append 管**session 级追加**；自定义字符串管**完全重造**（默认工具指导、安全指令、环境上下文三样全丢，须自行补齐）。